## Final Project: James Li

Importing stuff

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score, roc_curve, auc, precision_score, recall_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

Reading the CSVs.

In [ ]:
df1 = pd.read_csv('Training.csv')
df1['reviewText'] = df1['reviewText'].fillna('')
df1['summary'] = df1['summary'].fillna('')
df1['text'] = df1['summary'] + ' ' + df1['reviewText']

df2 = pd.read_csv('Test.csv')
df2['reviewText'] = df2['reviewText'].fillna('')
df2['summary'] = df2['summary'].fillna('')
df2['text'] = df2['summary'] + ' ' + df2['reviewText']

## Calculating optimal Macro-F1 score by iterating C values for logistic regression

In [ ]:
for cutoff in [1,2,3,4]:
    X_train = df1['text']
    y_train = (df1['overall'] > cutoff).astype(int)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])

    param_grid = {'tfidf__ngram_range': [(1,1),(1,2)],
                  'tfidf__min_df': [1,2,3,4,5],
                  'tfidf__max_df': [0.8,0.9,1.0],
                  'tfidf__max_features': [40000,45000,50000,55000],
                  'clf__C': np.linspace(0.1,1.5,20)}
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"cutoff={cutoff}:", grid.best_params_, grid.best_score_)

Optimal parameters: ngram_range = (1,2); max_df = 0.9; min_df=2; max_features=50000

cutoff=1: {'clf__C': 1.8} 0.786553613117714
cutoff=2: {'clf__C': np.float64(1.2777777777777777)} 0.8321426749040415
cutoff=3: {'clf__C': np.float64(1.0555555555555556)} 0.8485731303341092
cutoff=4: {'clf__C': 0.275} 0.8171640398250112

## Calculating optimal Macro-F1 score by iterating C values for SVM.

In [ ]:
for cutoff in [1,2,3,4]: 
    X = df1['text']
    y = (df1['overall'] > cutoff).astype(int)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(sublinear_tf=True)), 
        ('clf', LinearSVC(class_weight='balanced')),
    ])

    param_grid = {'tfidf__ngram_range': [(1,1),(1,2)],
                  'tfidf__min_df': [2,3,4,5],
                  'tfidf__max_df': [0.8,0.9,1.0],
                  'tfidf__max_features': [40000,45000,50000,55000],
                  'clf__C': np.linspace(0.1,0.3,11)}   

    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    grid.fit(X, y)
    print(f"cutoff={cutoff}:", grid.best_params_, grid.best_score_)

Optimal parameters: ngram_range = (1,2); max_df = 0.9; min_df=2; max_features=50000
cutoff=1: {'clf__C': np.float64(0.19)} 0.7907636279228369
cutoff=2: {'clf__C': np.float64(0.19)} 0.8331676511093447
cutoff=3: {'clf__C': np.float64(0.16)} 0.8501829313194962
cutoff=4: {'clf__C': np.float64(0.064)} 0.8220198815250367

## Calculating optimal Macro-F1 score by iterating alpha values of a Naive Bayes classifier

In [ ]:
for cutoff in [1,2,3,4]:
    X = df1['text']
    y = (df1['overall'] > cutoff).astype(int)

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(sublinear_tf=True)),
        ('clf', ComplementNB()),
    ])

    param_grid = {'tfidf__ngram_range': [(1,1),(1,2)],
                  'tfidf__min_df': [2,3,4,5],
                  'tfidf__max_df': [0.8,0.9,1.0],
                  'tfidf__max_features': [40000,45000,50000,55000],
                  'clf__alpha': np.linspace(0.1,0.3,11)}   

    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    grid.fit(X, y)
    print(f"cutoff={cutoff}:", grid.best_params_, grid.best_score_)



Optimal parameters: ngram_range = (1,2); max_df = 0.9; min_df=2; max_features=50000
cutoff=1: {'clf__alpha': np.float64(0.333265306122449)} 0.7909083484482169
cutoff=2: {'clf__alpha': np.float64(0.8181632653061224)} 0.8292316753197244
cutoff=3: {'clf__alpha': np.float64(0.4140816326530612)} 0.8407971028538279
cutoff=4: {'clf__alpha': np.float64(0.11102040816326529)} 0.8055565603398318

## Calculating accuracy, F1, AUC, confusion matrix, and plotting ROC for binary classifiers.

Creating functions to streamline the process. make_tfidf is tuned to the optimal parameters set from training. 

In [ ]:
def make_tfidf():
    return TfidfVectorizer(lowercase=True, sublinear_tf=True,
                           ngram_range=(1, 2), min_df=2, max_df = 0.9, max_features=50000)

def get_scores(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def run_classifier(name, make_clf):
    for cutoff in [1, 2, 3, 4]:
        X = df1['text']
        y = (df1['overall'] > cutoff).astype(int)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y)

        pipe = Pipeline([('tfidf', make_tfidf()), ('clf', make_clf(cutoff))])
        pipe.fit(X_tr, y_tr)

        y_pred = pipe.predict(X_val)
        y_score = get_scores(pipe, X_val)

        acc = accuracy_score(y_val, y_pred)
        mf1 = f1_score(y_val, y_pred, average='macro')
        auc = roc_auc_score(y_val, y_score)
        cm = confusion_matrix(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        print(f"\n=== {name} | cutoff={cutoff} ===")
        print(f"accuracy: {acc:.4f}  macro-F1: {mf1:.4f}  precision: {precision:.4f}  recall: {recall:.4f}  AUC: {auc:.4f}")
        print(f"confusion matrix [rows=true, cols=pred]:\n{cm}")

        fpr, tpr, _ = roc_curve(y_val, y_score)
        plt.figure(figsize=(5, 4))
        plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
        plt.xlabel('FPR'); plt.ylabel('TPR')
        plt.title(f'ROC — {name}, cutoff={cutoff}')
        plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

Logistic Regression

In [ ]:
logreg_C = {1: 1.8, 2: 1.27, 3: 1.06, 4: 0.275}
run_classifier('LogisticRegression',
    lambda c: LogisticRegression(C=logreg_C[c], max_iter=1000, class_weight='balanced'))

SVM

In [ ]:
svc_C = {1: 0.19, 2: 0.19, 3: 0.16, 4: 0.064}
run_classifier('LinearSVC',
    lambda c: LinearSVC(C=svc_C[c], class_weight='balanced'))

Naive Bayes

In [ ]:
nb_alpha = {1: 0.333, 2: 0.818, 3: 0.414, 4: 0.12}   
run_classifier('ComplementNB',
    lambda c: ComplementNB(alpha=nb_alpha[c]))

## Multiclass classification using Logistic Regression

Finding metrics

In [ ]:
classes = [1, 2, 3, 4, 5]
X = df1['text']
y = df1['overall']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, sublinear_tf=True, ngram_range=(1,2), min_df=2, max_features=50000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid = {'clf__C': np.linspace(0.5, 2.0, 8)}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print("best:", grid.best_params_, grid.best_score_)
model = grid.best_estimator_


y_pred = model.predict(X_val)                              
mf1 = f1_score(y_val, y_pred, average='macro')        
y_score = model.predict_proba(X_val)
y_bin = label_binarize(y_val, classes=classes)
acc = accuracy_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_score, multi_class='ovr', average='macro')
cm = confusion_matrix(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro')
recall = recall_score(y_val, y_pred, average='macro')

print(f"\n=== Logistic Regression (multiclass) ===")
print(f"accuracy: {acc:.4f}  macro-F1: {mf1:.4f}  precision: {precision:.4f}  recall: {recall:.4f}  AUC: {auc:.4f}")
print(f"confusion matrix [rows=true, cols=pred]:\n{cm}")

Plotting the ROCs for the 5 classes, plus the averaged 6th class

In [ ]:
fpr_grid = np.linspace(0, 1, 1000)   
interp_tprs = []

for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    plt.plot(fpr, tpr, label=f"class {c} (AUC={auc(fpr,tpr):.3f})")
    interp = np.interp(fpr_grid, fpr, tpr)
    interp[0] = 0.0        
    interp_tprs.append(interp)

mean_tpr = np.mean(interp_tprs, axis=0)
mean_tpr[-1] = 1.0           
macro_auc = auc(fpr_grid, mean_tpr)

plt.plot(fpr_grid, mean_tpr, color='black', linewidth=2.5, linestyle='--',
         label=f"macro-average (AUC={macro_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k:', alpha=0.4)   
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC — Logistic Regression')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Multiclass classification using SVC

Metrics

In [ ]:
classes = [1, 2, 3, 4, 5]

X = df1['text']
y = df1['overall']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, sublinear_tf=True, ngram_range=(1,2), min_df=2, max_features=50000)),
    ('clf', LinearSVC(class_weight='balanced'))
])

param_grid = {'clf__C': np.linspace(0.09, 0.15, 8)}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print("best:", grid.best_params_, grid.best_score_)
model = grid.best_estimator_

y_pred = model.predict(X_val)                              
mf1 = f1_score(y_val, y_pred, average='macro')        
df_scores = model.decision_function(X_val)
y_score = np.exp(df_scores) / np.exp(df_scores).sum(axis=1, keepdims=True)

y_bin = label_binarize(y_val, classes=classes)
acc = accuracy_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_score, multi_class='ovr', average='macro')
cm = confusion_matrix(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro')
recall = recall_score(y_val, y_pred, average='macro')

print(f"\n=== LinearSVC (multiclass) ===")
print(f"accuracy: {acc:.4f}  macro-F1: {mf1:.4f}  precision: {precision:.4f}  recall: {recall:.4f}  AUC: {auc:.4f}")
print(f"confusion matrix [rows=true, cols=pred]:\n{cm}")

ROC/AUC

In [ ]:
fpr_grid = np.linspace(0, 1, 1000)   
interp_tprs = []

for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    plt.plot(fpr, tpr, label=f"class {c} (AUC={auc(fpr,tpr):.3f})")
    interp = np.interp(fpr_grid, fpr, tpr)
    interp[0] = 0.0        
    interp_tprs.append(interp)

mean_tpr = np.mean(interp_tprs, axis=0)
mean_tpr[-1] = 1.0           
macro_auc = auc(fpr_grid, mean_tpr)

plt.plot(fpr_grid, mean_tpr, color='black', linewidth=2.5, linestyle='--',
         label=f"macro-average (AUC={macro_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k:', alpha=0.4)   # chance line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC — SVM')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Multiclass classification using Naive Bayes

Metrics

In [ ]:
classes = [1, 2, 3, 4, 5]

X = df1['text']
y = df1['overall']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, sublinear_tf=True, ngram_range=(1,2), min_df=2, max_features=50000)),
    ('clf', ComplementNB())
])

param_grid = {'clf__alpha': np.linspace(4.5, 4.8, 8)}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print("best:", grid.best_params_, grid.best_score_)
model = grid.best_estimator_

y_pred = model.predict(X_val)                              
mf1 = f1_score(y_val, y_pred, average='macro')        
y_score = model.predict_proba(X_val)
y_bin = label_binarize(y_val, classes=classes)
acc = accuracy_score(y_val, y_pred)
auc = roc_auc_score(y_val, y_score, multi_class='ovr', average='macro')
cm = confusion_matrix(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='macro')
recall = recall_score(y_val, y_pred, average='macro')

print(f"\n=== Naive Bayes (multiclass) ===")
print(f"accuracy: {acc:.4f}  macro-F1: {mf1:.4f}  precision: {precision:.4f}  recall: {recall:.4f}  AUC: {auc:.4f}")
print(f"confusion matrix [rows=true, cols=pred]:\n{cm}")

ROC/AUC

In [ ]:
fpr_grid = np.linspace(0, 1, 1000)   
interp_tprs = []

for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    plt.plot(fpr, tpr, label=f"class {c} (AUC={auc(fpr,tpr):.3f})")
    interp = np.interp(fpr_grid, fpr, tpr)
    interp[0] = 0.0        
    interp_tprs.append(interp)

mean_tpr = np.mean(interp_tprs, axis=0)
mean_tpr[-1] = 1.0           
macro_auc = auc(fpr_grid, mean_tpr)

plt.plot(fpr_grid, mean_tpr, color='black', linewidth=2.5, linestyle='--',
         label=f"macro-average (AUC={macro_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k:', alpha=0.4)   # chance line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC — Naive Bayes')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Predicting Labels for Test.csv

Best binary classifiers, Logistic Regression for cutoff 1 and 2, and LinearSVC for cutoff 3 and 4 as they had the highest macro-F1 score for each cutoff. Code saves predictions to Test_binary_predictions.csv, where predicted classes are put at the last 4 columns of the test dataset. 

In [ ]:
df_bc = pd.read_csv('Test.csv')
df_bc['reviewText'] = df_bc['reviewText'].fillna('')
df_bc['summary'] = df_bc['summary'].fillna('')
df_bc['text'] = df_bc['summary'] + ' ' + df_bc['reviewText']

best_clf = {
    1: LogisticRegression(max_iter=1000, class_weight='balanced', C=1.8),    
    2: LogisticRegression(max_iter=1000, class_weight='balanced', C=1.27),   
    3: LinearSVC(class_weight='balanced', C=0.16),                           
    4: LinearSVC(class_weight='balanced', C=0.064),                          
}

for cutoff in [1, 2, 3, 4]:
    y = (df1['overall'] > cutoff).astype(int)
    model = Pipeline([
        ('tfidf', make_tfidf()),
        ('clf', best_clf[cutoff])
    ])
    model.fit(df1['text'], y)
    df_bc[f'pred_cutoff_{cutoff}'] = model.predict(df_bc['text'])

df_bc.to_csv('Test_binary_predictions.csv', index=False)  

Best multiclass classifier, Logistic Regression with the highest Macro-F1 score. Code predictions to Test_predictions.csv, where predicted classes are put at the last columns of the test dataset. 

In [ ]:
df_mc = pd.read_csv('Test.csv')
df_mc['reviewText'] = df_mc['reviewText'].fillna('')
df_mc['summary'] = df_mc['summary'].fillna('')
df_mc['text'] = df_mc['summary'] + ' ' + df_mc['reviewText']

best_model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, sublinear_tf=True,
                              ngram_range=(1,2), min_df=2, max_features=50000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
best_model.fit(df1['text'], df1['overall'])

df_mc['overall'] = best_model.predict(df_mc['text'])
df_mc.drop(columns=['text']).to_csv('Test_predictions.csv', index=False)

print(df_mc['overall'].value_counts().sort_index())

## Clustering with K-means

We will be using k=6, as there are 6 product categories.

In [ ]:
for max_ft in [500, 1000, 2000, 5000]:
    for mn_df in [50,100,150,200,250]:
        vec = TfidfVectorizer(lowercase=True, max_features=max_ft, min_df=mn_df)
        X = vec.fit_transform(df2['text'])
        for n in [2, 3, 5, 10, 20, 50]:
            svd = TruncatedSVD(n_components=n, random_state=42)
            X_reduced = Normalizer().fit_transform(svd.fit_transform(X))

            km = KMeans(n_clusters=6, random_state=42, n_init=10)
            labels = km.fit_predict(X_reduced)

            sil = silhouette_score(X_reduced, labels)
            rand = adjusted_rand_score(df2['category'], labels)
            if sil>0.59:
                print(f"n_components={n}: silhouette={sil:.4f}  rand={rand:.4f} max_features={max_ft} min_df={mn_df}")
            